In [30]:
import os 
import numpy as np, pandas as pd
from glob import glob

from ifsoac_2 import *

In [31]:
def dimension_fractal(orb):
    """(orb.real, orb.imag)"""
    # pares = orb.view('(2,)float')  # ver complejos como par de reales
    # pares = np.array((orb.real, orb.imag)).T
    # scales = np.logspace(6, 15, num=16, endpoint=True, base=2)
    scales = np.logspace(0.01, 10, num=16, endpoint=False, base=2)
    ns = []
    for escala in scales:
        h, _, _ = np.histogram2d(orb[0], orb[1], bins=(int(escala), int(escala)))
        n = np.sum(h > 0)
        ns.append(n)
        if n >= orb.shape[1]:
            break
    coeffs = np.polyfit(np.log2(scales[:len(ns)]), np.log2(ns), 1)
    return coeffs[0]

In [35]:
root_folder = "C:/Users/brend/Documents/Recortes/"

# Carpeta raíz contiene las carpetas "DCL" y "control"
subfolders = ["DCL", "Control"]

results = []

for subfolder in subfolders:
    subfolder_path = os.path.join(root_folder, subfolder)
    label = '1' if subfolder == "DCL" else '0'
    if os.path.exists(subfolder_path):
        for folder in os.listdir(subfolder_path):
            folder_path = os.path.join(subfolder_path, folder)
            if os.path.isdir(folder_path):
                for file_path in glob(os.path.join(folder_path, "*.txt")):
                    # Verificar el tamaño del archivo antes de intentar leerlo
                    if os.path.getsize(file_path) == 0:
                        print(f"Archivo vacío: {file_path}. Ignorando...")
                        continue
                    folder_name = folder.split('_')  # Separar el nombre de la carpeta por el guion bajo
                    folder_name_before = folder_name[0] if len(folder_name) > 0 else ''  # Parte antes del guion bajo
                    folder_name_after = folder_name[1] if len(folder_name) > 1 else ''   # Parte después del guion bajo
                    file_name = os.path.splitext(os.path.basename(file_path))[0]  # Obtener el nombre del archivo sin la extensión
                    file_number = file_name.split('_')[-1]  # Separar el nombre del archivo por el guion bajo y obtener el último elemento
                    data = pd.read_csv(file_path, header=None).to_numpy()[5000:158600]
                    op = {"rotate":0, "cmap_ds":"CET_C1", "ventana_ds":1200, "cols_ds":4}
                    img_points = np.array(Ifsoac(data, op).jDC()).T
                    fractal_dimension = dimension_fractal(img_points)
                    results.append([label, folder_name_before, folder_name_after, file_number, fractal_dimension])

# Convertir los resultados en un DataFrame de Pandas
df = pd.DataFrame(results, columns=['Grupo', 'Participante', 'Lobulo', 'Minuto', 'Dimensión Fractal'])
print(df)



Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_C3\minuto_4.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_C3\minuto_5.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_C4\minuto_4.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_C4\minuto_5.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_CZ\minuto_4.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_CZ\minuto_5.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_EMG\minuto_4.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_EMG\minuto_5.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_F3\minuto_4.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_F3\minuto_5.txt. Ignorando...
Archivo vacío: C:/Users/brend/Documents/Recortes/Control\VC_F4\minuto_4.txt. Ignorando...
Archivo 

In [41]:
# Pivotar la tabla para tener cada lóbulo en una columna con sus dimensiones fractales
df_pivoted = df.pivot_table(index=['Grupo', 'Participante', 'Minuto'], columns='Lobulo', values='Dimensión Fractal', aggfunc='mean').reset_index()

print(df_pivoted)

Lobulo Grupo Participante Minuto        C3        C4        CZ       EMG  \
0          0           EM      1  1.529065  1.519571  1.497843  1.543680   
1          0           EM      2  1.513058  1.517424  1.464616  1.481137   
2          0           EM      3  1.457383  1.468861  1.406819  1.533749   
3          0           EM      4  1.488804  1.509383  1.413651  1.487632   
4          0           EM      5  1.443936  1.438641  1.428218  1.321638   
..       ...          ...    ...       ...       ...       ...       ...   
83         1           RR      1  1.433194  1.460543  1.439628       NaN   
84         1           RR      2  1.320163  1.312420  1.297008       NaN   
85         1           RR      3  1.455067  1.442145  1.463962       NaN   
86         1           RR      4  1.390421  1.409209  1.442896       NaN   
87         1           RR      5  1.475004  1.453029  1.472863       NaN   

Lobulo        F3        F4        F7  ...        O1        O2        P3  \
0       1.54

In [42]:
# Guardar el DataFrame pivotado en un archivo CSV
df_pivoted.to_csv('DM.csv', index=False)

print("Archivo CSV guardado exitosamente.")

Archivo CSV guardado exitosamente.
